# HISTOPANTUM colorectal MobileNetV2 comparison

Binary tumour versus non-tumour classification using the exact case assignment from Experiment 6. This notebook compares a frozen ImageNet-pretrained MobileNetV2 against partial fine-tuning from `block_14_expand`, with Batch Normalization frozen. Research and educational use only.

## Controlled protocol

Experiment 7 preserves the Exp 6 seed, 224 x 224 inputs, case-disjoint 28/6/6 assignment, augmentation, optimizer, learning rates, early stopping, threshold, and metrics. Only the backbone and its required preprocessing change. MobileNetV2 preprocessing is represented by a serializable `Rescaling` layer, avoiding the Lambda deserialization issue found in Exp 6.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from tensorflow import keras

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 15
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
EXPECTED_ASSIGNMENT_SHA256 = 'b8bca9f2baabca3e3bf3f118a321f8e65fa146f35858f597645aaeb1df85a37a'
OUTPUT_DIR = Path('/content/exp7_outputs') if Path('/content').exists() else Path('outputs')
DATASET_ROOT = Path(os.environ.get('HISTOPANTUM_COLON_ROOT', '/content/histopantum/histopantum/colon'))
DATASET_ARCHIVE = Path(os.environ.get('HISTOPANTUM_ARCHIVE', '/content/histopantum.zip'))

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f'Deterministic TensorFlow operations unavailable: {exc}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'python': sys.version, 'tensorflow': tf.__version__, 'sklearn': sklearn.__version__})


{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'tensorflow': '2.20.0', 'sklearn': '1.6.1'}


In [2]:
candidates = [
    DATASET_ROOT, Path('/content/histopantum/colon'), Path('/content/colon'),
    Path('data/histopantum/colon'),
]
DATASET_ROOT = next((path for path in candidates if path.is_dir()), DATASET_ROOT)
if not DATASET_ROOT.is_dir() and DATASET_ARCHIVE.is_file():
    extract_root = Path('/content/histopantum')
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ARCHIVE) as archive:
        archive.extractall(extract_root)
    candidates = [extract_root / 'histopantum' / 'colon', extract_root / 'colon']
    DATASET_ROOT = next((path for path in candidates if path.is_dir()), DATASET_ROOT)
required = [DATASET_ROOT / 'non-tumour', DATASET_ROOT / 'tumour']
assert all(path.is_dir() for path in required), f'Dataset not found at {DATASET_ROOT}'
print('Dataset root:', DATASET_ROOT.resolve())


Dataset root: /content/histopantum/histopantum/colon


In [3]:
EXP6_ASSIGNMENTS = {
    'TCGA-3L-AA1B':'train','TCGA-4N-A93T':'test','TCGA-4T-AA8H':'train',
    'TCGA-5M-AAT4':'train','TCGA-5M-AAT6':'train','TCGA-A6-2677':'validation',
    'TCGA-A6-A56B':'train','TCGA-AD-6548':'train','TCGA-AD-6889':'train',
    'TCGA-AD-A5EJ':'validation','TCGA-AG-3882':'train','TCGA-AG-4008':'train',
    'TCGA-AG-4015':'train','TCGA-AU-6004':'train','TCGA-AY-A54L':'test',
    'TCGA-AZ-5407':'train','TCGA-AZ-6608':'test','TCGA-CA-5796':'validation',
    'TCGA-CA-6715':'train','TCGA-CA-6718':'train','TCGA-CK-5915':'train',
    'TCGA-CK-6751':'validation','TCGA-CM-4748':'train','TCGA-D5-6538':'test',
    'TCGA-D5-6540':'train','TCGA-DM-A0XF':'test','TCGA-DM-A280':'validation',
    'TCGA-F4-6459':'train','TCGA-F4-6704':'train','TCGA-F4-6856':'train',
    'TCGA-G4-6315':'test','TCGA-G4-6322':'train','TCGA-NH-A5IV':'train',
    'TCGA-NH-A6GB':'validation','TCGA-NH-A6GC':'train','TCGA-QG-A5Z2':'train',
    'TCGA-QL-A97D':'train','TCGA-SS-A7HO':'train','TCGA-T9-A92H':'train',
    'TCGA-WS-AB45':'train',
}
assignment_csv = 'case_id,split\n' + ''.join(
    f'{case_id},{EXP6_ASSIGNMENTS[case_id]}\n' for case_id in sorted(EXP6_ASSIGNMENTS)
)
assignment_sha256 = hashlib.sha256(assignment_csv.encode()).hexdigest()
assert assignment_sha256 == EXPECTED_ASSIGNMENT_SHA256

def parse_patch(path: Path, label: int) -> dict:
    """Parse one HISTOPANTUM patch and attach its Exp 6 case split."""
    parts = path.stem.rsplit('_', 2)
    assert len(parts) == 3 and parts[1].isdigit() and parts[2].isdigit(), path.name
    slide_id, x, y = parts
    case_id = '-'.join(slide_id.split('-')[:3])
    assert case_id in EXP6_ASSIGNMENTS, f'Unexpected case: {case_id}'
    return {
        'path': str(path.resolve()), 'relative_path': path.relative_to(DATASET_ROOT).as_posix(),
        'label': label, 'class_name': 'tumour' if label else 'non-tumour',
        'case_id': case_id, 'slide_id': slide_id, 'x': int(x), 'y': int(y),
        'split': EXP6_ASSIGNMENTS[case_id],
    }

records = []
for class_name, label in [('non-tumour', 0), ('tumour', 1)]:
    paths = sorted((DATASET_ROOT / class_name).glob('*.jpg'))
    assert paths
    records.extend(parse_patch(path, label) for path in paths)
manifest = pd.DataFrame(records)
assert len(manifest) == 27_248 and manifest['relative_path'].nunique() == 27_248
assert manifest['case_id'].nunique() == 40 and manifest['slide_id'].nunique() == 40
assert manifest.groupby('case_id')['split'].nunique().max() == 1
summary = manifest.groupby('split').agg(
    patches=('label','size'), cases=('case_id','nunique'), slides=('slide_id','nunique'), tumour=('label','sum')
)
summary['non_tumour'] = summary['patches'] - summary['tumour']
display(summary.loc[['train','validation','test']])
assert summary.loc['train','patches'] == 19092
assert summary.loc['validation','patches'] == 3985
assert summary.loc['test','patches'] == 4171
manifest.drop(columns='path').to_csv(OUTPUT_DIR / 'split_manifest.csv', index=False)


,patches,cases,slides,tumour,non_tumour
split,,,,,
train,19092,28,28,11868,7224
validation,3985,6,6,2450,1535
test,4171,6,6,2631,1540


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path: tf.Tensor, label: tf.Tensor):
    """Decode one JPEG as a fixed-shape float32 RGB tensor."""
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    return tf.cast(image, tf.float32), tf.cast(label, tf.float32)

def make_dataset(part: pd.DataFrame, training: bool) -> tf.data.Dataset:
    """Create a finite deterministic dataset; augmentation stays in the model."""
    dataset = tf.data.Dataset.from_tensor_slices((part['path'].to_numpy(), part['label'].to_numpy()))
    if training:
        dataset = dataset.shuffle(len(part), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

parts = {name: manifest.loc[manifest['split'] == name].reset_index(drop=True) for name in ('train','validation','test')}
train_ds = make_dataset(parts['train'], True)
validation_ds = make_dataset(parts['validation'], False)
test_ds = make_dataset(parts['test'], False)
images, labels = next(iter(train_ds))
assert images.shape[1:] == (224,224,3) and labels.ndim == 1
print('Batch:', images.shape, labels.shape, float(tf.reduce_min(images)), float(tf.reduce_max(images)))


Batch: (32, 224, 224, 3) (32,) 0.0 255.0


In [5]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    keras.layers.RandomRotation(0.25, fill_mode='reflect', seed=SEED),
    keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED),
    keras.layers.RandomContrast(0.10, seed=SEED),
], name='training_augmentation')
backbone = keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE,3), pooling='avg'
)
backbone.trainable = False
inputs = keras.Input(shape=(*IMAGE_SIZE,3), name='image')
x = augmentation(inputs)
x = keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name='mobilenet_v2_preprocessing')(x)
x = backbone(x, training=False)
x = keras.layers.Dropout(0.30, seed=SEED)(x)
outputs = keras.layers.Dense(1, activation='sigmoid', name='tumour_probability')(x)
model = keras.Model(inputs, outputs, name='histopantum_crc_mobilenet_v2')

def compile_model(target: keras.Model, learning_rate: float) -> None:
    """Compile the binary classifier with the Exp 6 metric contract."""
    target.compile(
        optimizer=keras.optimizers.Adam(learning_rate), loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='roc_auc')],
    )

compile_model(model, HEAD_LEARNING_RATE)
probe = model(images[:2], training=False).numpy()
assert probe.shape == (2,1) and np.isfinite(probe).all() and ((probe >= 0) & (probe <= 1)).all()
print('Frozen trainable parameters:', sum(np.prod(v.shape) for v in model.trainable_weights))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Frozen trainable parameters: 1281


In [6]:
FROZEN_CHECKPOINT = OUTPUT_DIR / 'mobilenet_v2_frozen_best.keras'
def callbacks_for(checkpoint: Path) -> list[keras.callbacks.Callback]:
    """Build fresh validation-loss callbacks for one training phase."""
    return [
        keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_loss', save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1),
    ]
frozen_history = model.fit(
    train_ds, validation_data=validation_ds, epochs=HEAD_EPOCHS,
    callbacks=callbacks_for(FROZEN_CHECKPOINT), verbose=1,
)
pd.DataFrame(frozen_history.history).to_csv(OUTPUT_DIR / 'frozen_history.csv', index_label='epoch')
assert FROZEN_CHECKPOINT.is_file()


Epoch 1/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.8314 - loss: 0.3718 - precision: 0.8456 - recall: 0.8885 - roc_auc: 0.8943
Epoch 1: val_loss improved from None to 0.39169, saving model to /content/exp7_outputs/mobilenet_v2_frozen_best.keras

Epoch 1: finished saving model to /content/exp7_outputs/mobilenet_v2_frozen_best.keras
597/597 ━━━━━━━━━━━━━━━━━━━━ 72s 105ms/step - accuracy: 0.8896 - loss: 0.2661 - precision: 0.8974 - recall: 0.9285 - roc_auc: 0.9537 - val_accuracy: 0.8396 - val_loss: 0.3917 - val_precision: 0.9210 - val_recall: 0.8086 - val_roc_auc: 0.9212 - learning_rate: 0.0010
Epoch 2/5
596/597 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9301 - loss: 0.1828 - precision: 0.9377 - recall: 0.9512 - roc_auc: 0.9779
Epoch 2: val_loss did not improve from 0.39169
597/597 ━━━━━━━━━━━━━━━━━━━━ 60s 100ms/step - accuracy: 0.9307 - loss: 0.1808 - precision: 0.9390 - recall: 0.9502 - roc_auc: 0.9783 - val_accuracy: 0.8364 - val_loss: 0.4202 - val_precision: 0.9072 

In [7]:
model = keras.models.load_model(FROZEN_CHECKPOINT)
backbone = model.get_layer('mobilenetv2_1.00_224')
backbone.trainable = True
fine_tune_start = next(i for i, layer in enumerate(backbone.layers) if layer.name == 'block_14_expand')
for index, layer in enumerate(backbone.layers):
    layer.trainable = index >= fine_tune_start and not isinstance(layer, keras.layers.BatchNormalization)
trainable_backbone_layers = [layer.name for layer in backbone.layers if layer.trainable]
assert trainable_backbone_layers and trainable_backbone_layers[0] == 'block_14_expand'
assert not any(layer.trainable for layer in backbone.layers if isinstance(layer, keras.layers.BatchNormalization))
compile_model(model, FINE_TUNE_LEARNING_RATE)
trainable_parameters = int(sum(np.prod(v.shape) for v in model.trainable_weights))
total_parameters = int(model.count_params())
print('Fine-tuned backbone layers:', len(trainable_backbone_layers), '/', len(backbone.layers))
print('Trainable parameters:', trainable_parameters, '/', total_parameters, f'({100*trainable_parameters/total_parameters:.2f}%)')
FINE_CHECKPOINT = OUTPUT_DIR / 'mobilenet_v2_block14_best.keras'
fine_history = model.fit(
    train_ds, validation_data=validation_ds, epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks_for(FINE_CHECKPOINT), verbose=1,
)
pd.DataFrame(fine_history.history).to_csv(OUTPUT_DIR / 'fine_tune_history.csv', index_label='epoch')
assert FINE_CHECKPOINT.is_file()


Fine-tuned backbone layers: 20 / 155
Trainable parameters: 1512001 / 2259265 (66.92%)
Epoch 1/15
597/597 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.9417 - loss: 0.1515 - precision: 0.9501 - recall: 0.9569 - roc_auc: 0.9845
Epoch 1: val_loss improved from None to 0.38662, saving model to /content/exp7_outputs/mobilenet_v2_block14_best.keras

Epoch 1: finished saving model to /content/exp7_outputs/mobilenet_v2_block14_best.keras
597/597 ━━━━━━━━━━━━━━━━━━━━ 70s 105ms/step - accuracy: 0.9468 - loss: 0.1416 - precision: 0.9550 - recall: 0.9596 - roc_auc: 0.9862 - val_accuracy: 0.8700 - val_loss: 0.3866 - val_precision: 0.9282 - val_recall: 0.8547 - val_roc_auc: 0.9347 - learning_rate: 1.0000e-05
Epoch 2/15
596/597 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.9598 - loss: 0.1069 - precision: 0.9649 - recall: 0.9710 - roc_auc: 0.9920
Epoch 2: val_loss did not improve from 0.38662
597/597 ━━━━━━━━━━━━━━━━━━━━ 61s 102ms/step - accuracy: 0.9588 - loss: 0.1085 - precision: 0.9641 - recal

In [8]:
checkpoints = {'frozen': FROZEN_CHECKPOINT, 'block14_fine_tuned': FINE_CHECKPOINT}
validation_results = {}
for name, checkpoint in checkpoints.items():
    candidate = keras.models.load_model(checkpoint)
    validation_results[name] = candidate.evaluate(validation_ds, return_dict=True, verbose=0)
display(pd.DataFrame(validation_results).T)
selected_name = min(validation_results, key=lambda name: validation_results[name]['loss'])
SELECTED_CHECKPOINT = OUTPUT_DIR / 'mobilenet_v2_selected_best.keras'
shutil.copy2(checkpoints[selected_name], SELECTED_CHECKPOINT)
with (OUTPUT_DIR / 'validation_selection.json').open('w', encoding='utf-8') as handle:
    json.dump({'selected': selected_name, 'results': validation_results}, handle, indent=2)
print('Selected using validation loss only:', selected_name)


,accuracy,loss,precision,recall,roc_auc
frozen,0.856211,0.380686,0.919535,0.839592,0.930038
block14_fine_tuned,0.870013,0.386616,0.928191,0.854694,0.934692


Selected using validation loss only: frozen


In [9]:
def evaluate_checkpoint(name: str, checkpoint: Path) -> dict:
    """Evaluate one prespecified checkpoint and export patch/case evidence."""
    candidate = keras.models.load_model(checkpoint)
    probabilities = candidate.predict(test_ds, verbose=1).reshape(-1)
    y_true = parts['test']['label'].to_numpy(dtype=int)
    y_pred = (probabilities >= 0.5).astype(int)
    predictions = parts['test'][['relative_path','case_id','slide_id','label']].copy()
    predictions['probability'] = probabilities
    predictions['prediction'] = y_pred
    predictions.to_csv(OUTPUT_DIR / f'{name}_test_predictions.csv', index=False)
    pooled = {
        'accuracy': accuracy_score(y_true,y_pred), 'balanced_accuracy': balanced_accuracy_score(y_true,y_pred),
        'precision': precision_score(y_true,y_pred,zero_division=0),
        'recall': recall_score(y_true,y_pred,zero_division=0),
        'specificity': recall_score(y_true,y_pred,pos_label=0,zero_division=0),
        'f1': f1_score(y_true,y_pred,zero_division=0), 'roc_auc': roc_auc_score(y_true,probabilities),
        'patches': len(y_true), 'cases': int(predictions['case_id'].nunique()),
    }
    case_rows = []
    for case_id, group in predictions.groupby('case_id'):
        true = group['label'].to_numpy(dtype=int); pred = group['prediction'].to_numpy(dtype=int)
        prob = group['probability'].to_numpy()
        case_rows.append({
            'case_id':case_id, 'patches':len(group), 'accuracy':accuracy_score(true,pred),
            'balanced_accuracy':balanced_accuracy_score(true,pred),
            'f1':f1_score(true,pred,zero_division=0),
            'roc_auc':roc_auc_score(true,prob) if len(np.unique(true)) == 2 else np.nan,
        })
    case_metrics = pd.DataFrame(case_rows)
    case_metrics.to_csv(OUTPUT_DIR / f'{name}_case_metrics.csv', index=False)
    aggregate = case_metrics[['accuracy','balanced_accuracy','f1','roc_auc']].agg(['mean','std']).to_dict()
    case_macro = {metric:{stat:(None if pd.isna(value) else float(value)) for stat,value in stats.items()}
                  for metric,stats in aggregate.items()}
    result = {
        'checkpoint':checkpoint.name, 'threshold':0.5, 'pooled_patch_metrics':pooled,
        'case_macro_metrics':case_macro, 'confusion_matrix':confusion_matrix(y_true,y_pred).tolist(),
        'classification_report':classification_report(
            y_true,y_pred,target_names=['non-tumour','tumour'],output_dict=True,zero_division=0),
    }
    with (OUTPUT_DIR / f'{name}_test_metrics.json').open('w',encoding='utf-8') as handle:
        json.dump(result,handle,indent=2,allow_nan=False)
    return result

test_results = {name:evaluate_checkpoint(name,path) for name,path in checkpoints.items()}
comparison = pd.DataFrame({name:result['pooled_patch_metrics'] for name,result in test_results.items()}).T
comparison.to_csv(OUTPUT_DIR / 'test_model_comparison.csv', index_label='model')
display(comparison)
print('Deployment candidate selected before test evaluation:', selected_name)


131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 95ms/step
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 95ms/step


,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,patches,cases
frozen,0.951570,0.943840,0.950984,0.973394,0.914286,0.962059,0.986326,4171.0,6.0
block14_fine_tuned,0.959242,0.949787,0.951228,0.985937,0.913636,0.968272,0.990239,4171.0,6.0


Deployment candidate selected before test evaluation: frozen


In [10]:
def sha256_file(path: Path) -> str:
    """Return the lowercase SHA-256 digest of a file."""
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

selected_model = keras.models.load_model(SELECTED_CHECKPOINT)
selected_trainable_parameters = int(sum(np.prod(v.shape) for v in selected_model.trainable_weights))
model_manifest = {
    'experiment_id':'exp-7', 'dataset':'HISTOPANTUM colorectal subset',
    'task':'binary tumour versus non-tumour patch classification',
    'architecture':'MobileNetV2', 'grouping_unit':'TCGA case ID', 'seed':SEED,
    'exp6_assignment_sha256':assignment_sha256, 'selected_phase':selected_name,
    'model_file':SELECTED_CHECKPOINT.name, 'model_size_bytes':SELECTED_CHECKPOINT.stat().st_size,
    'model_sha256':sha256_file(SELECTED_CHECKPOINT), 'input_shape':[224,224,3],
    'preprocessing':'serializable Rescaling(1/127.5, offset=-1)', 'decision_threshold':0.5,
    'selected_model_trainable_parameters':selected_trainable_parameters,
    'fine_tuning_trainable_parameters':trainable_parameters, 'total_parameters':total_parameters,
    'fine_tuning_trainable_backbone_layers':len(trainable_backbone_layers),
    'backbone_layers':len(backbone.layers),
    'limitations':['Only 40 TCGA cases are available.','Patches within a case are correlated.',
                   'Internal colorectal evaluation only; not clinical or external validation.'],
}
with (OUTPUT_DIR / 'model_manifest.json').open('w',encoding='utf-8') as handle:
    json.dump(model_manifest,handle,indent=2)
compact_dir = Path('/content/exp7_compact_files') if Path('/content').exists() else Path('exp7_compact_files')
if compact_dir.exists(): shutil.rmtree(compact_dir)
compact_dir.mkdir(parents=True)
for artifact in OUTPUT_DIR.iterdir():
    if artifact.is_file() and artifact.suffix != '.keras': shutil.copy2(artifact,compact_dir/artifact.name)
archive_base = Path('/content/exp7_compact_evidence') if Path('/content').exists() else Path('exp7_compact_evidence')
archive = Path(shutil.make_archive(str(archive_base),'zip',compact_dir))
print('Selected model:',SELECTED_CHECKPOINT,model_manifest['model_sha256'])
print('Compact evidence:',archive,archive.stat().st_size,'bytes')
print('Download the selected model, compact ZIP, and executed notebook before ending Colab.')


Selected model: /content/exp7_outputs/mobilenet_v2_selected_best.keras 4d10344cd5b481d936a2471178b390cc2987c353b44fdcabae6af4a152a98c55
Compact evidence: /content/exp7_compact_evidence.zip 302007 bytes
Download the selected model, compact ZIP, and executed notebook before ending Colab.


In [11]:
!zip -r /content/exp7_outputs.zip /content/exp7_outputs

  adding: content/exp7_outputs/ (stored 0%)
  adding: content/exp7_outputs/mobilenet_v2_frozen_best.keras (deflated 12%)
  adding: content/exp7_outputs/block14_fine_tuned_case_metrics.csv (deflated 45%)
  adding: content/exp7_outputs/frozen_history.csv (deflated 51%)
  adding: content/exp7_outputs/split_manifest.csv (deflated 93%)
  adding: content/exp7_outputs/validation_selection.json (deflated 51%)
  adding: content/exp7_outputs/block14_fine_tuned_test_predictions.csv (deflated 90%)
  adding: content/exp7_outputs/test_model_comparison.csv (deflated 40%)
  adding: content/exp7_outputs/block14_fine_tuned_test_metrics.json (deflated 64%)
  adding: content/exp7_outputs/fine_tune_history.csv (deflated 49%)
  adding: content/exp7_outputs/model_manifest.json (deflated 39%)
  adding: content/exp7_outputs/frozen_test_predictions.csv (deflated 90%)
  adding: content/exp7_outputs/mobilenet_v2_selected_best.keras (deflated 12%)
  adding: content/exp7_outputs/frozen_case_metrics.csv (deflated 44